# 04. パーティション進化（Spark）

## 隠しパーティション

Iceberg では `months(pickup_at)` のように **列の値を変換した結果** でパーティションを切れます。
パーティション用の列（`pickup_month` など）を別に持つ必要はなく、クエリも `WHERE pickup_at >= ...` と普通に書くだけで、関係ないパーティションのファイルは読み飛ばされます。
利用者がパーティションの存在を意識しなくてよいので「隠しパーティション」と呼ばれます。

## パーティション進化

データが増えてパーティションの切り方を変えたくなったとき、Iceberg は **既存のデータを書き直さずに** 切り方を変えられます。
変更後に書いたファイルだけが新しい切り方になり、古いファイルは古い切り方のまま共存します。

- テーブル: `handson.taxi_part_spark`
- 事前に `make data` でデータを取得しておく

## 準備: SparkSession を作る

接続設定は `spark-defaults.conf` にあるので、ここでは何も指定しません（01 と同じ）。

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("04_partition_evolution").getOrCreate()

def sql(query):
    """SQL を実行し、結果があれば表示する"""
    df = spark.sql(query)
    if df.columns:
        df.show(truncate=False)

## 1. 月単位のパーティションでテーブルを作る

`PARTITIONED BY (months(pickup_at))` を指定します。
実データには範囲外の日時（2008 年など）が混ざっているので、その月の乗車だけに絞って入れます。

In [ ]:
sql("DROP TABLE IF EXISTS handson.taxi_part_spark PURGE")
sql("""
CREATE TABLE handson.taxi_part_spark (
    vendor_id     INT,
    pickup_at     TIMESTAMP_NTZ,
    trip_distance DOUBLE,
    total_amount  DOUBLE
) USING iceberg
PARTITIONED BY (months(pickup_at))
""")

def load_month(month, start, end):
    """指定した月のファイルから、その月の乗車だけを追記する"""
    spark.read.parquet(f"/workspace/data/yellow_tripdata_{month}.parquet").createOrReplaceTempView("src")
    sql(f"""
    INSERT INTO handson.taxi_part_spark
    SELECT VendorID, tpep_pickup_datetime, trip_distance, total_amount FROM src
    WHERE tpep_pickup_datetime >= '{start}' AND tpep_pickup_datetime < '{end}'
    """)

load_month("2024-12", "2024-12-01", "2025-01-01")

メタデータテーブル `partitions` で、パーティションごとの行数とファイル数を確認します。
パーティションの値は「1970 年 1 月からの月数」で表されます（659 = 2024 年 12 月）。

In [ ]:
sql("SELECT partition, spec_id, record_count, file_count FROM handson.taxi_part_spark.partitions ORDER BY partition")

## 2. 隠しパーティションでクエリする

パーティションを意識せず、`pickup_at` で絞り込むだけです。
Spark UI（http://localhost:4040 ）の SQL タブでこのクエリを開くと、スキャンの詳細にパーティションで絞り込まれたことが表示されます。

In [ ]:
sql("""
SELECT count(*) AS trips, round(avg(total_amount), 2) AS avg_total
FROM handson.taxi_part_spark
WHERE pickup_at >= '2024-12-24' AND pickup_at < '2024-12-26'
""")

## 3. パーティションの切り方を変える（月 → 日）

`REPLACE PARTITION FIELD` で、月単位から日単位に変えます。この操作はメタデータの変更だけで、既存のファイルには触れません。

In [ ]:
# パーティション仕様を変える ALTER は、カタログ名から完全に書く（lakehouse.handson.xxx）
sql("ALTER TABLE lakehouse.handson.taxi_part_spark REPLACE PARTITION FIELD pickup_at_month WITH days(pickup_at)")
load_month("2025-01", "2025-01-01", "2025-02-01")

パーティションの一覧を見ると、2024-12 は月単位（`spec_id = 0`）のまま、2025-01 は日単位（`spec_id = 1`）で 31 個に分かれています。
RustFS のコンソールで `warehouse/handson/taxi_part_spark.../data/` を開くと、`pickup_at_month=...` と `pickup_at_day=...` のディレクトリが並んでいます。
`spec_id` はパーティション仕様（切り方）の番号で、ファイルごとに「どの切り方で書かれたか」が記録されています。

In [ ]:
sql("""
SELECT spec_id, count(*) AS partitions, sum(record_count) AS rows, sum(file_count) AS files
FROM handson.taxi_part_spark.partitions
GROUP BY spec_id ORDER BY spec_id
""")
# ファイルのパスには、どの切り方で書かれたかがディレクトリ名として表れる
sql("""
SELECT spec_id, regexp_extract(file_path, 'pickup_at_[a-z]+=[^/]+', 0) AS partition_dir, record_count
FROM handson.taxi_part_spark.files
ORDER BY spec_id, partition_dir LIMIT 5
""")

## 4. 切り方が混在していても普通にクエリできる

2 つの切り方にまたがる範囲でも、クエリの書き方は変わりません。Iceberg がファイルごとの切り方に合わせて絞り込みます。

In [ ]:
sql("""
SELECT date_format(pickup_at, 'yyyy-MM') AS month, count(*) AS trips
FROM handson.taxi_part_spark
WHERE pickup_at >= '2024-12-30' AND pickup_at < '2025-01-03'
GROUP BY 1 ORDER BY 1
""")

## まとめ

- `months(col)` や `days(col)` などの変換でパーティションを切れる。クエリでパーティションを意識する必要はない（隠しパーティション）
- パーティションの切り方は後から変えられ、既存のデータは書き直されない
- 古い切り方のファイルと新しい切り方のファイルが共存し、ファイルごとに `spec_id` で区別される
- 古いデータも新しい切り方に揃えたいときは、07 の Compaction（`rewrite_data_files`）で書き直す